<a href="https://colab.research.google.com/github/ironspiritjeff/hf-transformer-experiments/blob/main/Copy_FoodNotFood.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# 1. Import necessary packages
import pprint
from pathlib import Path
import numpy as np
import torch
import datasets
from transformers import pipeline
from transformers import TrainingArguments, Trainer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
try: import evaluate
except ModuleNotFoundError:
  !pip install -U evaluate
  import evaluate

# 2. Set up variables for model training and saving pipeline
DATASET_NAME = 'mrdbourke/learn_hf_food_not_food_image_captions'
MODEL_NAME = 'distilbert/distilbert-base-uncased'
MODEL_SAVE_DIR_NAME = 'models2/learn_hf_food_not_food_text_classifier-distilbert-base-uncased'

# 3. Create a directory for saving models
print(f'[INFO] Creating directory for saving models: {MODEL_SAVE_DIR_NAME}')
model_save_dir = Path(MODEL_SAVE_DIR_NAME)
model_save_dir.mkdir(parents=True, exist_ok=True)

# 4. Load and preprocess dataset from Hugging Face Hub
print(f'Downloading dataset from HuggingFace Hub, name: {DATASET_NAME}')
dataset = datasets.load_dataset(DATASET_NAME)

id2label = {0: 'not_food', 1: 'food'}
label2id = {'not_food': 0, 'food': 1}

# Create a function to map IDs to labels in the dataset
def map_labels_to_number(example):
  example['label'] = label2id[example['label']]
  return example

# Map our preprocessing function to our dataset
dataset = dataset['train'].map(map_labels_to_number)

# Split the dataset into train/test sets
dataset = dataset.train_test_split(test_size=0.2, seed=42)
dataset

# 5. Import a tokenizer and map it to our dataset
print(f'Tokenizing text for model training with tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=MODEL_NAME,
                                          use_fast=True)

# Create a function to turn text into numbers using the above tokenizer
def tokenize_text(examples):
  return tokenizer(examples['text'],
                   padding=True,
                   truncation=True)

tokenized_dataset = dataset.map(function=tokenize_text,
                                batched=True,
                                batch_size=1000)
tokenized_dataset

# 6. Set up an evaluation metric
accuracy_metric = evaluate.load('accuracy')

def compute_accuracy(predictions_and_labels):
  predictions, labels = predictions_and_labels
  # (model will output logits in the form ([[item_1, item_2, item_3], [item_1, item_2, item_3]])), depending on the number of classes you have.
  # But we want to compare labels which are in the format ([0,0,0,0,1])
  if len(predictions.shape) >= 2:
    predictions = np.argmax(predictions, axis=1) # reduce the items to a single value
  return accuracy_metric.compute(predictions=predictions, references=labels)

# 7. Set up a model
print(f'Loading model: {MODEL_NAME}')
model = AutoModelForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path=MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)
print(f'Model loading complete!')

# Set up TrainingArguments (hyperparameters)
training_args = TrainingArguments(
    output_dir=model_save_dir,
    learning_rate=0.0001,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=3,
    use_cpu=False,
    seed=42,
    load_best_model_at_end=True,
    logging_strategy='epoch',
    report_to='none',
    push_to_hub=False,
    hub_private_repo=False
)

# Create Trainer Instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset = tokenized_dataset['train'],
    eval_dataset = tokenized_dataset['test'],
   ### tokenizer=tokenizer, (Gemini tells me this line shouldn't be included)
    compute_metrics=compute_accuracy
)

trainer

# 8. Train the model
print(f'[INFO] Commencing model training...')
results = trainer.train()

# 9. Save the trained model to a local directory
print(f'[INFO] Model training complete, saving to local path: {model_save_dir}')
trainer.save_model(output_dir=model_save_dir)

# 10. Push the model to the Hugging Face hub
print(f'[INFO] Uploading model to Hugging Face Hub...')
model_upload_url = trainer.push_to_hub(
    commit_message='Uploading food not food text classifier model (putting it all together)'
)
print(f'[INFO] Model upload complete, model available at: {model_upload_url}')

# 11. Evaluate the model on the test data
print(f'[INFO] Performing evaluation on test dataset...')
predictions_all = trainer.predict(tokenized_dataset['test'])
prediction_values = predictions_all.predictions
prediction_metrics = predictions_all.metrics

print(f'[INFO] prediction metrics on test data:')
pprint.pprint(prediction_metrics)

[INFO] Creating directory for saving models: models2/learn_hf_food_not_food_text_classifier-distilbert-base-uncased
Tokenizing text for model training with tokenizer: distilbert/distilbert-base-uncased
Loading model: distilbert/distilbert-base-uncased


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loading complete!
[INFO] Commencing model training...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.338834,0.041322,1.000000
2,0.019267,0.005754,1.000000
3,0.003719,0.002047,1.000000
4,0.001743,0.001152,1.000000
5,0.001047,0.000819,1.000000
6,0.000819,0.000670,1.000000
7,0.000706,0.000593,1.000000
8,0.000640,0.000550,1.000000
9,0.000582,0.000528,1.000000
10,0.000584,0.000520,1.000000


...

New Data Upload               : |          |  0.00B /  0.00B            

  ...uncased/model.safetensors:  49%|####9     |  132MB /  268MB            

  ...uncased/training_args.bin:  15%|#4        |   776B / 5.20kB            

[INFO] Model upload complete, model available at: https://huggingface.co/ironspiritjeff/learn_hf_food_not_food_text_classifier-distilbert-base-uncased/commit/dddac0ee55860e6d35a4d9acd753419f5bcb355e
[INFO] Performing evaluation on test dataset...


[INFO] prediction metrics on test data:
{'test_accuracy': 1.0,
 'test_loss': 0.0005204931367188692,
 'test_runtime': 0.0915,
 'test_samples_per_second': 546.257,
 'test_steps_per_second': 21.85}


In [14]:
# 12. Make sure the model works by testing it on a custom sample
from transformers import pipeline
food_not_food_classifier = pipeline(task='text-classification',
                                    model=model_save_dir,
                                    device=torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu'),
                                    top_k=1,
                                    batch_size=32
                                    )
food_not_food_classifier('berries and taco salad')

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[[{'label': 'not_food', 'score': 0.9660535454750061}]]